# qcconvert：Qiskit ⇄ CUDA-Q
建議 Runtime → Change runtime type → T4 GPU；CPU 可用 `qpp-cpu`。如果先在 CPU 安裝後再換 GPU，請重新安裝 CUDA-Q（安裝時依 CUDA 版本選 cu12／cu13）。
**Qiskit 原始碼輸入會執行輸入的程式碼**，只執行可信任內容。

## GPU 檢查（GPU check）

In [ ]:
import subprocess
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True, check=False) if __import__('shutil').which('nvidia-smi') else None
print(gpu.stdout if gpu and gpu.returncode == 0 else '沒有可用 GPU，改用 CPU')

## 安裝（installation）及上傳 zip
Colab 可先將 `qcconvert_colab.zip` 上傳至 `/content/`；在本機測試可設定 `QCCONVERT_SKIP_INSTALL=1`。

In [ ]:
import os, sys, pathlib, subprocess, zipfile
if os.getenv('QCCONVERT_SKIP_INSTALL') != '1':
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'cudaq', 'qiskit', 'ipywidgets'], check=True)
    archive = pathlib.Path('/content/qcconvert_colab.zip')
    if archive.exists():
        with zipfile.ZipFile(archive) as zf:
            zf.extractall('/content')
        sys.path.insert(0, '/content')
    elif 'google.colab' in sys.modules:
        print('請用 google.colab.files.upload() 上傳 qcconvert_colab.zip，然後重新執行本儲存格')
    else:
        print('非 Colab 環境；使用既有本機套件')
    if 'google.colab' in sys.modules:
        from google.colab import output
        if hasattr(output, 'enable_custom_widget_manager'):
            output.enable_custom_widget_manager()
else:
    print('跳過安裝（skip installation）')

## 環境與 API 範例（Bell／GHZ／參數化電路）

In [ ]:
import cudaq, qiskit, qcconvert
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
print('cudaq:', cudaq.__version__, 'qiskit:', qiskit.__version__, 'target:', qcconvert.pick_target())
bell = QuantumCircuit(2)
bell.h(0); bell.cx(0, 1)
ghz = QuantumCircuit(3)
ghz.h(0); ghz.cx(0, 1); ghz.cx(1, 2)
theta = Parameter('theta')
parametric = QuantumCircuit(1)
parametric.ry(theta, 0)
for label, circuit in [('Bell', bell), ('GHZ', ghz)]:
    converted = qcconvert.qiskit_to_cudaq(circuit)
    print(label, converted.code, qcconvert.compare(circuit, converted.to_kernel()), sep='\n')
bound = parametric.assign_parameters({theta: 0.3})
print('參數化電路', qcconvert.qiskit_to_cudaq(parametric).code, qcconvert.compare(bound), sep='\n')

## Cell magics：兩個方向的轉換

In [ ]:
%load_ext qcconvert

In [ ]:
%%qiskit2cudaq --run --name bell_kernel
from qiskit import QuantumCircuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

In [ ]:
%%cudaq2qiskit --name restored
import cudaq
@cudaq.kernel
def kernel():
    q = cudaq.qvector(2)
    h(q[0])
    x.ctrl(q[0], q[1])

## 雙欄編輯器（widget）

In [ ]:
app = qcconvert.show()
print('雙欄編輯器已預填 Bell 範例')

## 選用 GPU 加速比較
有 GPU 時可自行建立 20+ qubit GHZ，比對 `nvidia` 與 `qpp-cpu` 的耗時；CPU runtime 跳過。

In [ ]:
if cudaq.num_available_gpus() > 0:
    large = QuantumCircuit(21)
    large.h(0)
    for i in range(20): large.cx(i, i + 1)
    for target in ('nvidia', 'qpp-cpu'):
        os.environ['QCCONVERT_TARGET'] = target
        print(target, qcconvert.compare(large, shots=100).elapsed)
    os.environ.pop('QCCONVERT_TARGET', None)
else:
    print('沒有 GPU，跳過加速比較')

## Open in Colab 徽章（badge）
目前沒有 GitHub repository，因此不放無效的 Open in Colab 連結；日後有 repo 再補徽章。